In [1]:
source('../../../R/preamble.R')

freqplot = function(df, spacing = 0.0){
    ggplot(df) + aes(gen, freq + spacing*dense_rank(pos), col = pos, group = paste0(ref, pos, alt, wave)) + 
    geom_line(linewidth = 0.1) + geom_point(size = 0.3) +
    facet_wrap(~pop, ncol = 4) +
    labs(x = 'Timepoint', y = 'Frequency')
    }

In [2]:
d_eff = read.tsv('../../../datasets/genomics/04_gene_attribution/intermediate/d_eff.tsv')

# Add a human-readable label for each variant (vid)

Remove the old label v_string (pos_ref_alt_delta).

Make a geno_string (pos_ref_alt) so variations in repeat numbers are all pooled in the same geno_string.

Add a label of the form context_number where number is the rank of the variant by order of apparition and pos as a tiebreaker.

This means that the variants detected at the beginning will keep the same ID as new generations are added.

In [3]:
d_labeled =
    d_eff %>%

    # Stupid edge case variants that pop up just once at freq ~0.1 at the exact same time, frequency and position and this breaks my tiebreaker system and you can't do dense_rank on multiple things so it's all incredibly complicated and I'm so tired of this so I will just DESTROY THIS STUPID LITTLE VARIANT and not even report it in the material and methods and humanity will be better off anyways
    filter(pos != 1039766) %>% 

    # Get the variant's time of apparition + freq as tiebreaker
    group_by(pop, chrom, pos, ref, alt, wave) %>% 
    mutate(
        geno_string = paste(chrom, pos, ref, alt, sep = '_'),
        freq_at_apparition = freq[gen == apparition], # lower is better
        tiebreaker = (pos + 1 - freq_at_apparition*0.9) / 5e6, # pos then freq at apparition as tiebreaker, should be between 0 and 1
        score = apparition + tiebreaker
        ) %>%

    # The score of a variant is the lowest score it has anywhere
    group_by(geno_string) %>%
    mutate(
        min_score = min(score)
        ) %>%

    # Rank the variants in the same context by their order of apparition
    group_by(context) %>% 
    mutate(
        variant_rank = dense_rank(min_score)
        ) %>%
    ungroup() %>% 

    # Build identifiers
    mutate(
        vid = paste(context, variant_rank, sep = '_'), # for unique genotypes
        tid = paste(pop, vid, wave, sep = '_') # for unique freq trajectories
        ) %>%
    select(-c(geno_string, v_string, score, min_score, variant_rank))

In [4]:
# Split main experiment from the rest
d_labeled %>% 
    filter(nchar(pop) == 1) %>% 
    write.tsv('../../../datasets/genomics/04_gene_attribution/results/d_labeled.tsv')

d_labeled %>% 
    filter(nchar(pop) != 1) %>% 
    write.tsv('../../../datasets/genomics/04_gene_attribution/results/d_labeled_replays.tsv')

# Generate a variant catalogue

**v_all: all detected variants for each population, including those from cross-contaminations.**

Use this to know which variants are present in the lineages in their current state.

In [5]:
v_all =
d_labeled %>%

    # Don't count mutations in the prophage's genome
    filter(chrom == 'U00096') %>% 

    # To get the most frequent version of REP length changes
    group_by(pop, origin, context, effect, chrom, pos, ref, alt, wave, type, delta, cat, read_pop) %>%
    mutate(
        n = sum(freq)
        ) %>% 
    arrange(pop, origin, context, chrom, pos, ref, alt, wave, n) %>% 

    # Delta summary statistics
    group_by(pop, origin, context, chrom, pos, ref, alt, wave) %>% 
    mutate(
        min_delta = min(delta), max_delta = max(delta),
        edits = paste(unique(edits), collapse = ', ')
    ) %>%
    filter(row_number() == 1) %>%  # only keep the most frequent version
    ungroup() %>%

    select(pop, origin, vid, tid, context, effect, chrom, pos, ref, alt, wave, 
           type, cat, codon_ref, codon_alt, aa_ref, aa_pos, aa_alt,
           locus, rep_len, rep_nt, m_delta = delta, min_delta, max_delta, 
           max_f, gen = apparition, fixation, strand, target_type,
           product, edits, read_pop) %>%
    arrange(pop, gen, chrom, pos)

In [6]:
# Don't double-count the initial mutations that are present everywhere, are these are from an early cross-contamination event
initial_variants =
d_labeled %>%

    # Pick tids to be removed
    group_by(vid) %>% 
    filter(
        nchar(pop) == 1,
        gen == 1, freq > 0, # Present in first timepoint
        n_distinct(origin) > 1 # Present in multiple lines
    ) %>% 

    arrange(vid, -max_f) %>% # so the row with highest frequency is first

    #group_by(vid) %>%
    #filter(row_number() > 1) %>%  # keep the first row, so the variant is counted only once
    ungroup()

# Label the initial variants by setting their origin to a "I" population
v_all[v_all$tid %in% initial_variants$tid, 'origin'] = 'I'

**v: unique variants**

This only counts unique variant apparition events (and the population in which they appeared). Use this to do statistics on which variants tend to be selected for.

In [7]:
# Make a filtered version without cross-contaminations. This is what we will use for statistics.
v =
v_all %>%
    filter(
        pop == origin,
    )

v %>% 
    # Main experiment
    filter(nchar(pop) == 1) %>% 
    write.tsv('../../../datasets/genomics/04_gene_attribution/results/v.tsv')

v %>% 
    # Main experiment
    filter(nchar(pop) != 1) %>% 
    write.tsv('../../../datasets/genomics/04_gene_attribution/results/v_replays.tsv')

In [8]:
initial_variants %>% 
    distinct(vid, pos, ref, alt) %>% 
    write.tsv('../../../datasets/genomics/04_gene_attribution/results/initial_variants')